In [ ]:
import os
import cv2
import numpy as np
import time
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from skimage.feature import hog

DATA_DIR = "./PlantDoc-Split"

def load_features(split_type='train', img_size=(64, 64)):
    """
    Görüntüleri okur, HOG ve Renk özelliklerini çıkarır.
    """
    path = os.path.join(DATA_DIR, split_type)
    X_hog = []
    X_color = []
    y = []
    
    if not os.path.exists(path):
        print(f"HATA: {path} bulunamadı! Önce veri setini ayırma kodunu çalıştır.")
        return None, None, None

    classes = sorted(os.listdir(path))
    print(f"[{split_type.upper()}] verisi yükleniyor (Sınıf sayısı: {len(classes)})...")
    
    total_images = 0
    start_time = time.time()
    
    for label, class_name in enumerate(classes):
        class_dir = os.path.join(path, class_name)
        if not os.path.isdir(class_dir): continue
        
        for img_name in os.listdir(class_dir):
            img_path = os.path.join(class_dir, img_name)
            img = cv2.imread(img_path)
            if img is None: continue
            
            img = cv2.resize(img, img_size)
            
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            feat_hog = hog(gray, orientations=9, pixels_per_cell=(8,8), 
                           cells_per_block=(2,2), block_norm='L2-Hys', visualize=False)
            
            hist = cv2.calcHist([img], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256])
            feat_color = cv2.normalize(hist, hist).flatten()
            
            X_hog.append(feat_hog)
            X_color.append(feat_color)
            y.append(label)
            total_images += 1
            
    print(f"-> {total_images} resim işlendi. Süre: {time.time()-start_time:.1f} sn")
    return np.array(X_hog), np.array(X_color), np.array(y)


print("Veriler hazırlanıyor...")
X_train_h, X_train_c, y_train = load_features('train')
X_test_h, X_test_c, y_test = load_features('test')

results = {}

print("\n--- Model 1: SVM (HOG) Eğitiliyor... ---")
svm = SVC(kernel='linear', C=1.0)
svm.fit(X_train_h, y_train)
y_pred_svm = svm.predict(X_test_h)

results['SVM'] = {
    'Accuracy': accuracy_score(y_test, y_pred_svm),
    'Macro-F1': f1_score(y_test, y_pred_svm, average='macro')
}
print(f"✅ SVM Bitti! Acc: {results['SVM']['Accuracy']:.4f}, F1: {results['SVM']['Macro-F1']:.4f}")

print("\n--- Model 2: Random Forest (Renk) Eğitiliyor... ---")
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_c, y_train)
y_pred_rf = rf.predict(X_test_c)

results['RF'] = {
    'Accuracy': accuracy_score(y_test, y_pred_rf),
    'Macro-F1': f1_score(y_test, y_pred_rf, average='macro')
}
print(f"✅ RF Bitti! Acc: {results['RF']['Accuracy']:.4f}, F1: {results['RF']['Macro-F1']:.4f}")

print("\n--- RAPOR İÇİN TABLO VERİLERİ ---")
print(f"| Klasik ML | HOG + SVM | %{results['SVM']['Accuracy']*100:.1f} | {results['SVM']['Macro-F1']:.2f} |")
print(f"| Klasik ML | Color + RF| %{results['RF']['Accuracy']*100:.1f} | {results['RF']['Macro-F1']:.2f} |")

Veriler hazırlanıyor...
[TRAIN] verisi yükleniyor (Sınıf sayısı: 27)...
-> 2324 resim işlendi. Süre: 72.7 sn
[TEST] verisi yükleniyor (Sınıf sayısı: 27)...
-> 304 resim işlendi. Süre: 8.0 sn

--- Model 1: SVM (HOG) Eğitiliyor... ---
✅ SVM Bitti! Acc: 0.1217, F1: 0.0991

--- Model 2: Random Forest (Renk) Eğitiliyor... ---
✅ RF Bitti! Acc: 0.3520, F1: 0.2559

--- RAPOR İÇİN TABLO VERİLERİ ---
| Klasik ML | HOG + SVM | %12.2 | 0.10 |
| Klasik ML | Color + RF| %35.2 | 0.26 |
